In [ ]:
from openai import OpenAI
import json

client = OpenAI()
messages = []


In [5]:
def get_weather(city):
    return "33 degrees celcius."


FUNCTION_MAP = {
    "get_weather": get_weather,
}

In [ ]:
from openai.types.chat import ChatCompletionMessage


TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "A function to get the weather of a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The name of the city to get the weather of.",
                    }
                },
                "required": ["city"],
            },
        },
    }
]

def process_ai_response(message: ChatCompletionMessage):

    # tool_call이 존재할 경우 message에 툴콜 정보를 저장.
    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )

        # tool_call이 존재할 경우 해당 툴콜을 실행.
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with {arguments}")

            # 문자열인 형태인 JSON을 딕셔너리로 변환. ex) "{"city":"seoul"}" -> {"city": "seoul"}
            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)
            # **를 사용하여 딕셔너리를 풀어서 함수에 전달. ex) {"city": "seoul"} -> city="seoul"
            result = function_to_run(**arguments)

            print(f"Ran {function_name} with args {arguments} for a result of {result}")
            
            # 함수 실행 결과를 메모리에 저장.
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": result,
                }
            )
        # AI가 tool_call 실행으로 인해 추가된 새로운 messages를 대화에서 볼 수 있게 함.
        call_ai()
    # tool_call이 존재하지 않을 경우 일반적인 대화 흐름.
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")


def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)

In [ ]:
while True:
    message = input("Send a message to LLM...")
    if message == "quit":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        call_ai()

User: 내이름은 헨리야
AI: 안녕하세요, 헨리! 어떻게 도와드릴까요?
User: 오늘 한국날씨 몇도야?
Calling function: get_weather with {"city":"서울"}
Ran get_weather with args {'city': '서울'} for a result of 33 degrees celcius.
AI: 오늘 서울의 날씨는 33도입니다. 더운 날씨에 유의하세요! 다른 도움이 필요하신가요?
User: 아니야 없어 고마워
AI: 천만에요! 필요하실 때 언제든지 말씀해 주세요. 좋은 하루 되세요!
